In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

def prepare_data(series, window_size=10):
    # 1. Transformation en binaire (1 si > 2.0, sinon 0)
    binary_data = (series >= 2.0).astype(int).values
    
    X, y = [], []
    for i in range(len(binary_data) - window_size):
        X.append(binary_data[i:i + window_size])
        y.append(binary_data[i + window_size])
    
    # Reshape pour LSTM: (samples, time_steps, features)
    return np.array(X).reshape(-1, window_size, 1), np.array(y)
def train_lstm_binary(X_train, y_train):
    model = Sequential([
        # input_shape = (window_size, 1 feature)
        LSTM(50, activation='relu', input_shape=(X_train.shape[1], 1)),
        Dense(25, activation='relu'),
        Dense(1, activation='sigmoid') # Sortie entre 0 et 1 (probabilité)
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
    return model

In [5]:
import pandas as pd
def load_data(file_path):
    data = pd.read_csv(file_path)
    data['multiplier'] = pd.to_numeric(data['multiplier'], errors='coerce')
    multiplier_data = data['multiplier'].dropna()
    return data, multiplier_data

In [40]:
model = train_lstm_binary(*prepare_data(load_data('crash_history.csv')[1], window_size=10))

d:\scraping\stake\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [41]:
def predict_next_round(model, last_10_results, window_size=10):
    # 1. Convertir les 10 derniers crashs en binaire (1 si > 2.0, sinon 0)
    binary_sequence = (np.array(last_10_results) >= 2.0).astype(int)
    
    # 2. Redimensionner pour le modèle (1 échantillon, 10 étapes, 1 caractéristique)
    input_data = binary_sequence.reshape((1, window_size, 1))
    
    # 3. Obtenir la probabilité
    prediction_prob = model.predict(input_data, verbose=0)[0][0]
    
    # 4. Décision
    if prediction_prob > 0.5:
        return "MISEZ (Probabilité: {:.2f}%)".format(prediction_prob * 100)
    else:
        return "ATTENDEZ (Probabilité: {:.2f}%)".format(prediction_prob * 100)

In [42]:
last_10_results = pd.read_csv('crash_history.csv')['multiplier'].tail(10).values.astype(float).tolist()
print(predict_next_round(model, last_10_results, window_size=10))

ATTENDEZ (Probabilité: 49.35%)


In [231]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, log_loss

# 1. Préparation des données
def prepare_crash_data(series, window_size=10):
    # Transformation en binaire : 1 si > 2.0, sinon 0
    binary_series = (series >= 2.0).astype(int)
    
    X = []
    y = []
    
    # Création de fenêtres coulissantes (le passé pour prédire le futur)
    for i in range(len(binary_series) - window_size):
        X.append(binary_series[i:i + window_size])
        y.append(binary_series[i + window_size])
        
    return np.array(X), np.array(y)

# --- SIMULATION DE DONNÉES (Remplace par tes vraies données ici) ---
# data = pd.Series([1.5, 2.5, 1.1, 4.0, 1.8, 3.2, ...])
data = load_data('crash_history.csv')[1]
# -----------------------------------------------------------------

# 2. Paramètres
WINDOW_SIZE = 3  # On regarde les 5 derniers coups pour prédire le 6ème

# 3. Transformation
X, y = prepare_crash_data(data, window_size=WINDOW_SIZE)

# 4. Division Entraînement / Test (80% pour apprendre, 20% pour vérifier)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 5. Création et entraînement du modèle
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Évaluation
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"--- RÉSULTATS ---")
print(f"Précision du modèle (Accuracy): {accuracy * 100:.2f}%")
print("Log loss: {:.4f}".format(log_loss(y_test, model.predict_proba(X_test)[:,1])))
print("\nDétails du rapport :")
print(classification_report(y_test, y_pred))

# 7. Fonction pour prédire le prochain tour réel
def predict_next(last_results):
    last_binary = (np.array(last_results[-WINDOW_SIZE:]) >= 2.0).astype(int)
    prediction = model.predict([last_binary])
    proba = model.predict_proba([last_binary])[0][1]
    return "GAGNANT (>2x)" if prediction[0] == 1 else "PERDANT (<2x)", proba

# Exemple d'usage sur les 10 derniers résultats réels
# derniers_crashs = [1.2, 3.5, 0.5, 2.1, 1.8, 1.1, 4.5, 2.2, 1.9, 1.3]
# print(predict_next(derniers_crashs))

--- RÉSULTATS ---
Précision du modèle (Accuracy): 51.63%
Log loss: 0.6932

Détails du rapport :
              precision    recall  f1-score   support

           0       0.52      0.76      0.62       778
           1       0.50      0.26      0.34       723

    accuracy                           0.52      1501
   macro avg       0.51      0.51      0.48      1501
weighted avg       0.51      0.52      0.48      1501



In [237]:
x= 1000
perte_succession = 0
perte_succession_max = 0
parier = False
for i in range(x):
    last_10_results = pd.read_csv('crash_history.csv')['multiplier'][:-(x-i)].tail(9).values.astype(float).tolist()
    if last_10_results[-1] >= 2.0 and parier:
        perte_succession = 0
        parier = False
    elif parier:
        perte_succession += 1
        perte_succession_max = max(perte_succession_max, perte_succession)
    
    predict_next_round = predict_next(last_10_results)
    if predict_next_round[1]>= 0.5:
        parier = True
    
    

In [238]:
perte_succession_max

9

## Avec features

In [224]:
from sklearn.metrics import log_loss
def enhance_features(series, window=5):
    df = pd.DataFrame(series, columns=['multiplier'])
    
    # Indicateur 1: Moyenne mobile des 5 derniers crashs
    df['sma_5'] = df['multiplier'].rolling(window=5).mean()
    
    # Indicateur 2: Écart-type (volatilité)
    df['std_5'] = df['multiplier'].rolling(window=5).std()
    
    # Indicateur 3: Compteur de "Rouges" consécutifs (< 2x)
    is_red = (df['multiplier'] < 2.0).astype(int)
    df['red_streak'] = is_red.groupby((is_red != is_red.shift()).cumsum()).cumcount() + 1
    # On ne garde le streak que si le dernier était rouge
    df['red_streak'] = df['red_streak'] * is_red 

    # Cibles (Targets)
    df['target_2x'] = (df['multiplier'] >= 2.0).astype(int)
    df['target_4x'] = (df['multiplier'] >= 4.0).astype(int)
    
    return df.dropna()

# Préparation
data = load_data('crash_history.csv')[0]
df_enriched = enhance_features(data)
features = ['sma_5', 'std_5', 'red_streak']
X = df_enriched[features]
y = df_enriched['target_2x'].shift(-1).fillna(0) # On veut prédire le PROCHAIN

# Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Modèle
model_2x = RandomForestClassifier(n_estimators=150, max_depth=10)
model_2x.fit(X_train, y_train)

# 6. Évaluation
y_pred = model_2x.predict(X_test)
y_class_prob = model_2x.predict_proba(X_test)[:,1]
accuracy = accuracy_score(y_test, y_pred)

print(f"--- RÉSULTATS ---")
print(f"Précision du modèle (Accuracy): {accuracy * 100:.2f}%")
print("Log loss: {:.4f}".format(log_loss(y_test, y_class_prob)))
print("\nDétails du rapport :")
print(classification_report(y_test, y_pred))

def get_prediction_now(model, last_raw_data):
    # 1. Calculer les indicateurs sur les derniers crashs reçus
    # (On simule ce que le modèle a appris : SMA, Volatilité, etc.)
    last_5 = last_raw_data[-5:]
    sma = np.mean(last_5)
    std = np.std(last_5)
    
    # Compter la série de rouges actuelle
    red_streak = 0
    for val in reversed(last_raw_data):
        if val < 2.0: red_streak += 1
        else: break
            
    # 2. Créer le vecteur de données pour le modèle
    features = np.array([[sma, std, red_streak]])
    
    # 3. Prédire la probabilité (0 à 1)
    proba_2x = model.predict_proba(features)[0][1] # Probabilité pour > 2x
    
    return proba_2x

--- RÉSULTATS ---
Précision du modèle (Accuracy): 50.43%
Log loss: 0.7002

Détails du rapport :
              precision    recall  f1-score   support

         0.0       0.52      0.54      0.53       779
         1.0       0.48      0.46      0.47       722

    accuracy                           0.50      1501
   macro avg       0.50      0.50      0.50      1501
weighted avg       0.50      0.50      0.50      1501



In [241]:
# x= 40
# for i in range(x):
#     last_10_results = pd.read_csv('crash_history.csv')['multiplier'][:-(x-i)].tail(9).values.astype(float).tolist()
#     print(last_10_results[-1], "\n")
#     print(get_prediction_now(model_2x, last_10_results))

x= 1000
perte_succession = 0
perte_succession_max = 0
parier = False
for i in range(x):
    last_10_results = pd.read_csv('crash_history.csv')['multiplier'][:-(x-i)].tail(9).values.astype(float).tolist()
    if last_10_results[-1] >= 2.0 and parier:
        perte_succession = 0
        parier = False
    elif parier:
        perte_succession += 1
        perte_succession_max = max(perte_succession_max, perte_succession)
    
    predict_next_round = get_prediction_now(model_2x, last_10_results)
    if predict_next_round>= 0.5:
        parier = True

d:\scraping\stake\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\scraping\stake\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\scraping\stake\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\scraping\stake\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\scraping\stake\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\sc

9

In [218]:
from stakepred import AdvancedPredictor

predictor = AdvancedPredictor()

In [219]:
predictor.train_models_from_csv('crash_history.csv')

{'accuracy': 0.4806666666666667,
 'log_loss': 0.6989674864768005,
 'mse': 29424.706485083436,
 'n_samples': 7499}

In [90]:
predictor.save_models()

In [ ]:
# x = 40
# for i in range(x):
#     predictor.history.clear()
#     predictor.history.extend([("",float(x)) for x in pd.read_csv('crash_history.csv')['multiplier'][:-(x-i)].tail(20).values])
#     print(predictor.history[-1][1],"\n")
#     # print(predictor.predict_next_multiplier())
#     print(predictor.predict_next_safety())

x= 1000
perte_succession = 0
perte_succession_max = 0
parier = False
for i in range(x):
    predictor.history.clear()
    predictor.history.extend([("",float(x)) for x in pd.read_csv('crash_history.csv')['multiplier'][:-(x-i)].tail(20).values])
    if predictor.history[-1][1] >= 2.0 and parier:
        perte_succession = 0
        parier = False
    elif parier:
        perte_succession += 1
        perte_succession_max = max(perte_succession_max, perte_succession)
    
    predict_next_round = predictor.predict_next_safety()
    if predict_next_round>= 0.5:
        parier = True

TypeError: '>=' not supported between instances of 'tuple' and 'float'

4.627962443483337
0.5026899401727101
